# AI-Powered Telecom Customer Retention Platform

**Student:** Maurizio Pinto  
**Program:** AI Master's — Udacity / Woolf University  
**Date:** May 2026  

## Project Overview

This project integrates methods from three prior capstone projects into an end-to-end telecom customer retention system:

| Component | Prior Project | Methods |
|---|---|---|
| Statistical Analysis | Project 2 — Statistical Analysis | Chi-squared test, Welch's t-test, descriptive statistics |
| ML Churn Prediction | Project 3 — ML Model Design | Logistic Regression, Random Forest, feature importance |
| Agentic Retention Advisor | Project 6 — Agentic AI | ReAct loop, OpenAI function calling, safeguards |

**Pipeline flow:** Statistical Analysis → ML Prediction → Risk-Adjusted CLV → Agentic Retention Action

## Setup

In [1]:
import json
import os
import time
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix, ConfusionMatrixDisplay
)

from openai import OpenAI
from dotenv import load_dotenv

sns.set_theme(style='whitegrid')
%matplotlib inline
np.random.seed(42)

## Data Loading & Inspection

In [2]:
df = pd.read_csv('dataset/WA_Fn-UseC_-Telco-Customer-Churn.csv')
print(f'Dataset shape: {df.shape}')
print(f'Columns: {df.shape[1]}, Rows: {df.shape[0]}')
df.head()

Dataset shape: (7043, 21)
Columns: 21, Rows: 7043


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   str    
 1   gender            7043 non-null   str    
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   str    
 4   Dependents        7043 non-null   str    
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   str    
 7   MultipleLines     7043 non-null   str    
 8   InternetService   7043 non-null   str    
 9   OnlineSecurity    7043 non-null   str    
 10  OnlineBackup      7043 non-null   str    
 11  DeviceProtection  7043 non-null   str    
 12  TechSupport       7043 non-null   str    
 13  StreamingTV       7043 non-null   str    
 14  StreamingMovies   7043 non-null   str    
 15  Contract          7043 non-null   str    
 16  PaperlessBilling  7043 non-null   str    
 17  Paymen

In [4]:
print('Missing values:')
print(df.isnull().sum())
print(f'\nTotal missing: {df.isnull().sum().sum()}')

Missing values:
customerID          0
gender              0
SeniorCitizen       0
Partner             0
Dependents          0
tenure              0
PhoneService        0
MultipleLines       0
InternetService     0
OnlineSecurity      0
OnlineBackup        0
DeviceProtection    0
TechSupport         0
StreamingTV         0
StreamingMovies     0
Contract            0
PaperlessBilling    0
PaymentMethod       0
MonthlyCharges      0
TotalCharges        0
Churn               0
dtype: int64

Total missing: 0


In [5]:
# Class distribution
churn_counts = df['Churn'].value_counts()
churn_pct = df['Churn'].value_counts(normalize=True) * 100
print('Churn distribution:')
print(f'  No Churn:  {churn_counts["No"]:,} ({churn_pct["No"]:.1f}%)')
print(f'  Churn:     {churn_counts["Yes"]:,} ({churn_pct["Yes"]:.1f}%)')
print(f'  Churn rate: {churn_pct["Yes"]:.1f}%')

Churn distribution:
  No Churn:  5,174 (73.5%)
  Churn:     1,869 (26.5%)
  Churn rate: 26.5%
